# Uncertainty Quantification for Magnetic Field Prediction

This chapter explores advanced uncertainty quantification techniques for deep learning models in electromagnetic field prediction. Understanding uncertainty is crucial for reliable deployment in engineering applications where confidence in predictions directly impacts safety and decision-making.

## Learning Objectives

After completing this notebook, you will understand:

- **Sources of Uncertainty**: Aleatoric vs epistemic uncertainty in physics-based models
- **Bayesian Neural Networks**: Probabilistic deep learning for uncertainty estimation
- **Monte Carlo Dropout**: Practical uncertainty quantification method
- **Deep Ensembles**: Model averaging for robust uncertainty estimates
- **Heteroscedastic Uncertainty**: Input-dependent noise modeling
- **Calibration and Validation**: Ensuring reliable uncertainty estimates

## Types of Uncertainty in Physics-Based Models

### 1. Aleatoric Uncertainty (Irreducible)

Aleatoric uncertainty arises from inherent randomness in the data:

- **Measurement noise**: Sensor errors, experimental limitations
- **Numerical discretization**: Finite element mesh errors
- **Material property variations**: Manufacturing tolerances, temperature effects

**Mathematical Representation**:

$$p(y|x, \theta) = \mathcal{N}(f(x, \theta), \sigma^2(x))$$

### 2. Epistemic Uncertainty (Reducible)

Epistemic uncertainty stems from model limitations:

- **Model architecture**: Capacity constraints
- **Training data**: Limited or biased samples
- **Parameter estimation**: Non-convex optimization landscape
- **Out-of-distribution**: Unseen parameter combinations

**Mathematical Representation**:

$$p(y|x, \mathcal{D}) = \int p(y|x, \theta) p(\theta|\mathcal{D}) d\theta$$

### 3. Total Uncertainty

The total predictive uncertainty combines both sources:

$$\text{Var}[y|x] = \mathbb{E}[\text{Var}[y|x, \theta]] + \text{Var}[\mathbb{E}[y|x, \theta]]$$

where the first term is aleatoric and the second is epistemic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

## Monte Carlo Dropout: Practical Uncertainty Quantification

### 1. Theory Behind MC Dropout

Monte Carlo Dropout provides a practical approximation to Bayesian inference:

**Key Insight**: Dropout during inference creates different thinned network architectures, providing an ensemble effect.

**Mathematical Foundation**:

- Each dropout mask samples a sub-network from the full network
- Multiple forward passes approximate the posterior predictive distribution
- Computational efficiency: single model, multiple predictions

### 2. Implementation Strategy

**Training Phase**:
- Standard dropout with probability $p$
- Optimize weights using standard backpropagation

**Inference Phase**:
- Enable dropout during forward pass
- Perform $T$ stochastic forward passes
- Compute statistics from predictions

**Predictive Statistics**:
- Mean: $\hat{y} = \frac{1}{T}\sum_{t=1}^{T} f_t(x)$
- Variance: $\sigma^2 = \frac{1}{T}\sum_{t=1}^{T} f_t^2(x) - \hat{y}^2$
- Confidence intervals: $\hat{y} \pm z_{\alpha/2}\sigma$

In [ ]:
# Monte Carlo Dropout Implementation
class MCDropoutMagneticFieldNet(nn.Module):
    """Magnetic Field Prediction Network with Monte Carlo Dropout"""
    
    def __init__(self, input_dim=3, hidden_dims=[64, 32, 16], output_dim=1, dropout_rate=0.2):
        super(MCDropoutMagneticFieldNet, self).__init__()
        
        self.dropout_rate = dropout_rate
        
        # Build network layers
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            if i < len(hidden_dims) - 1:  # Don't add dropout after last hidden layer
                layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)
    
    def predict_with_uncertainty(self, x, n_samples=100):
        """Generate predictions with uncertainty estimates"""
        self.train()  # Enable dropout during inference
        
        predictions = []
        
        with torch.no_grad():
            for _ in range(n_samples):
                pred = self(x)
                predictions.append(pred)
        
        predictions = torch.stack(predictions)
        
        # Calculate statistics
        mean = predictions.mean(dim=0)
        std = predictions.std(dim=0)
        
        # Calculate percentiles for confidence intervals
        percentiles_5 = torch.quantile(predictions, 0.05, dim=0)
        percentiles_95 = torch.quantile(predictions, 0.95, dim=0)
        
        return {
            'mean': mean,
            'std': std,
            'percentile_5': percentiles_5,
            'percentile_95': percentiles_95,
            'all_predictions': predictions
        }

# Demonstration of MC Dropout
def demonstrate_mc_dropout():
    """Demonstrate MC Dropout on synthetic magnetic field data"""
    
    print("🎲 Monte Carlo Dropout Demonstration")
    print("=" * 50)
    
    # Generate synthetic data
    torch.manual_seed(42)
    n_samples = 1000
    
    # Input: geometry, material, excitation parameters
    X = torch.randn(n_samples, 3)
    
    # True function (non-linear with noise)
    y_true = 0.5 * X[:, 0]**2 + 0.3 * X[:, 1] * X[:, 2] + 0.1 * torch.randn(n_samples)
    y_true = y_true.unsqueeze(1)
    
    # Add heteroscedastic noise
    noise_std = 0.1 + 0.05 * torch.abs(X[:, 0])
    y = y_true + noise_std.unsqueeze(1) * torch.randn(n_samples, 1)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Create and train model
    model = MCDropoutMagneticFieldNet(input_dim=3, hidden_dims=[32, 16], dropout_rate=0.2)
    
    # Training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    # Training loop
    model.train()
    epochs = 200
    
    print("\n🏃 Training MC Dropout model...")
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    
    # Generate uncertainty estimates
    print("\n📊 Generating uncertainty estimates...")
    uncertainty_results = model.predict_with_uncertainty(X_test[:100], n_samples=50)
    
    # Calculate metrics
    mean_pred = uncertainty_results['mean']
    std_pred = uncertainty_results['std']
    true_values = y_test[:100]
    
    mse = torch.mean((mean_pred - true_values)**2)
    coverage_95 = torch.mean(
        ((true_values >= uncertainty_results['percentile_5']) & 
         (true_values <= uncertainty_results['percentile_95'])).float()
    )
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Monte Carlo Dropout: Uncertainty Quantification Results', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Predictions vs True values
    sample_idx = torch.arange(50)
    axes[0, 0].plot(sample_idx.numpy(), true_values[:50].numpy(), 'k-o', 
                    label='True Values', markersize=4)
    axes[0, 0].plot(sample_idx.numpy(), mean_pred[:50].numpy(), 'b-s', 
                    label='Mean Prediction', markersize=4)
    axes[0, 0].fill_between(sample_idx.numpy(), 
                           uncertainty_results['percentile_5'][:50].numpy().flatten(),
                           uncertainty_results['percentile_95'][:50].numpy().flatten(),
                           alpha=0.3, color='blue', label='95% CI')
    axes[0, 0].set_xlabel('Sample Index')
    axes[0, 0].set_ylabel('Magnetic Field')
    axes[0, 0].set_title('Predictions with Confidence Intervals')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Uncertainty vs Error
    errors = torch.abs(mean_pred - true_values)
    axes[0, 1].scatter(std_pred.numpy().flatten(), errors.numpy().flatten(), alpha=0.6)
    axes[0, 1].set_xlabel('Predicted Uncertainty (σ)')
    axes[0, 1].set_ylabel('Absolute Error')
    axes[0, 1].set_title('Uncertainty vs Prediction Error')
    axes[0, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print(f"\n📈 Results Summary:")
    print(f"   MSE: {mse.item():.4f}")
    print(f"   95% Coverage: {coverage_95.item():.1%}")
    print(f"   Mean Uncertainty: {std_pred.mean().item():.4f}")
    
    return model, uncertainty_results

# Run demonstration
mc_model, mc_results = demonstrate_mc_dropout()

## Deep Ensembles: Robust Uncertainty Estimation

### 1. Ensemble Theory

Deep ensembles combine multiple independently trained neural networks:

**Mathematical Foundation**:

$$f_{ensemble}(x) = \frac{1}{M}\sum_{m=1}^{M} f_m(x)$$

where $f_m$ are individual models with different initializations and training data subsets.

### 2. Uncertainty Decomposition

Deep ensembles naturally capture both uncertainty types:

**Predictive Variance**:

$$\text{Var}[y|x] = \underbrace{\frac{1}{M}\sum_{m=1}^{M}\text{Var}[y_m|x]}_{\text{Aleatoric}} + \underbrace{\frac{1}{M}\sum_{m=1}^{M}(\mathbb{E}[y_m|x] - \mathbb{E}[y|x])^2}_{\text{Epistemic}}$$

### 3. Training Strategies

**Bagging (Bootstrap Aggregating)**:
- Train each model on different bootstrap samples
- Reduces variance and improves stability

**Snapshot Ensembles**:
- Collect models from different training epochs
- Computational efficiency

**Diverse Architectures**:
- Different network architectures
- Different hyperparameters
- Maximum diversity

In [ ]:
# Deep Ensemble Implementation
class DeepEnsemble:
    """Deep Ensemble for magnetic field prediction"""
    
    def __init__(self, n_models=5, input_dim=3, hidden_dims=[32, 16], output_dim=1):
        self.n_models = n_models
        self.models = []
        
        for _ in range(n_models):
            model = nn.Sequential(
                nn.Linear(input_dim, hidden_dims[0]),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dims[0], hidden_dims[1]),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dims[1], output_dim)
            )
            self.models.append(model)
    
    def train_ensemble(self, X_train, y_train, epochs=200, lr=0.01):
        """Train all models in the ensemble"""
        print(f"🏃 Training ensemble with {self.n_models} models...")
        
        for i, model in enumerate(self.models):
            print(f"  Training model {i+1}/{self.n_models}...")
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            criterion = nn.MSELoss()
            
            # Bootstrap sampling
            indices = torch.randint(0, len(X_train), (len(X_train),))
            X_bootstrap = X_train[indices]
            y_bootstrap = y_train[indices]
            
            model.train()
            for epoch in range(epochs):
                optimizer.zero_grad()
                outputs = model(X_bootstrap)
                loss = criterion(outputs, y_bootstrap)
                loss.backward()
                optimizer.step()
                
                if (epoch + 1) % 50 == 0:
                    print(f"    Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    
    def eval(self):
        """Set all models to evaluation mode"""
        for model in self.models:
            model.eval()
    
    def predict_with_uncertainty(self, x):
        """Generate ensemble predictions with uncertainty"""
        self.eval()
        
        predictions = []
        
        with torch.no_grad():
            for model in self.models:
                pred = model(x)
                predictions.append(pred)
        
        predictions = torch.stack(predictions)
        
        # Calculate ensemble statistics
        mean = predictions.mean(dim=0)
        std = predictions.std(dim=0)
        
        return {
            'mean': mean,
            'std': std,
            'all_predictions': predictions
        }

# Demonstrate Deep Ensembles
def demonstrate_deep_ensembles():
    """Demonstrate Deep Ensemble on magnetic field prediction"""
    
    print("🌊 Deep Ensemble Demonstration")
    print("=" * 50)
    
    # Generate synthetic data
    torch.manual_seed(123)
    n_samples = 800
    
    # More complex function for ensemble demonstration
    X = torch.randn(n_samples, 3)
    y_true = torch.sin(X[:, 0]) * X[:, 1] + 0.5 * X[:, 2]**2 + 0.1 * torch.randn(n_samples)
    y_true = y_true.unsqueeze(1)
    
    # Add heteroscedastic noise
    noise_std = 0.05 + 0.1 * torch.abs(X[:, 1])
    y = y_true + noise_std.unsqueeze(1) * torch.randn(n_samples, 1)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Create and train ensemble
    ensemble = DeepEnsemble(n_models=5, input_dim=3, hidden_dims=[32, 16])
    ensemble.train_ensemble(X_train, y_train, epochs=150)
    
    # Generate predictions
    print("\n📊 Generating ensemble predictions...")
    ensemble_results = ensemble.predict_with_uncertainty(X_test[:100])
    
    # Calculate metrics
    mean_pred = ensemble_results['mean']
    std_pred = ensemble_results['std']
    true_values = y_test[:100]
    
    mse = torch.mean((mean_pred - true_values)**2)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Deep Ensemble: Uncertainty Quantification Results', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Ensemble mean with confidence intervals
    sample_idx = torch.arange(30)
    axes[0, 0].plot(sample_idx.numpy(), true_values[:30].numpy(), 'k-o', 
                    label='True Values', markersize=4)
    axes[0, 0].plot(sample_idx.numpy(), mean_pred[:30].numpy(), 'b-s', 
                    label='Ensemble Mean', markersize=4)
    axes[0, 0].fill_between(sample_idx.numpy(), 
                           (mean_pred[:30] - 2*std_pred[:30]).numpy().flatten(),
                           (mean_pred[:30] + 2*std_pred[:30]).numpy().flatten(),
                           alpha=0.3, color='blue', label='95% CI')
    axes[0, 0].set_xlabel('Sample Index')
    axes[0, 0].set_ylabel('Magnetic Field')
    axes[0, 0].set_title('Ensemble Prediction with Uncertainty')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Uncertainty vs Error
    errors = torch.abs(mean_pred - true_values)
    axes[0, 1].scatter(std_pred.numpy().flatten(), errors.numpy().flatten(), alpha=0.6)
    axes[0, 1].set_xlabel('Ensemble Uncertainty (σ)')
    axes[0, 1].set_ylabel('Absolute Error')
    axes[0, 1].set_title('Uncertainty vs Prediction Error')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Individual model predictions
    individual_predictions = ensemble_results['all_predictions']
    for i in range(min(3, individual_predictions.shape[0])):
        axes[1, 0].plot(sample_idx.numpy(), individual_predictions[i, :30].numpy(), 
                        alpha=0.6, label=f'Model {i+1}')
    axes[1, 0].plot(sample_idx.numpy(), true_values[:30].numpy(), 'k-o', 
                    label='True Values', markersize=4, linewidth=2)
    axes[1, 0].set_xlabel('Sample Index')
    axes[1, 0].set_ylabel('Magnetic Field')
    axes[1, 0].set_title('Individual Model Predictions')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Uncertainty distribution
    axes[1, 1].hist(std_pred.numpy().flatten(), bins=20, alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Ensemble Uncertainty (σ)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Uncertainty Distribution')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print(f"\n📈 Ensemble Results Summary:")
    print(f"   Ensemble MSE: {mse.item():.4f}")
    print(f"   Mean Uncertainty: {std_pred.mean().item():.4f}")
    print(f"   Uncertainty Range: [{std_pred.min().item():.4f}, {std_pred.max().item():.4f}]")
    
    return ensemble

# Run demonstration
ensemble_model = demonstrate_deep_ensembles()

## Summary and Implementation Guidelines

### ✅ **Key Takeaways**

This chapter provides a comprehensive framework for uncertainty quantification in magnetic field prediction:

#### 1. **Theoretical Foundation**

- **Aleatoric vs Epistemic**: Understanding fundamental uncertainty sources
- **Bayesian Framework**: Principled approach to uncertainty quantification
- **Mathematical Rigor**: Proper treatment of probability distributions

#### 2. **Practical Methods**

- **MC Dropout**: Easy implementation, good for prototyping
- **Deep Ensembles**: Robust uncertainty, ideal for production
- **Bayesian Neural Networks**: Theoretically sound, research-oriented
- **Heteroscedastic Models**: Input-dependent noise modeling

#### 3. **Engineering Applications**

- **Safety-Critical Systems**: Conservative uncertainty estimates
- **Real-Time Applications**: Balance accuracy and computational efficiency
- **Research and Development**: Comprehensive uncertainty analysis

### 📊 **Implementation Roadmap**

**Phase 1: Prototyping**

1. Start with MC Dropout for quick implementation
2. Validate uncertainty calibration
3. Establish baseline performance metrics

**Phase 2: Production**

1. Implement Deep Ensembles for robustness
2. Add calibration monitoring
3. Deploy with uncertainty-aware decision making

### 🎯 **Best Practices**

1. **Always Validate**: Use calibration metrics and reliability diagrams
2. **Consider Application Context**: Choose methods based on deployment requirements
3. **Monitor Continuously**: Track uncertainty quality over time
4. **Physical Constraints**: Ensure uncertainty estimates respect physical laws
5. **Documentation**: Clearly communicate uncertainty limitations to stakeholders

### 🚀 **Next Steps**

With uncertainty quantification capabilities in place:

1. **Physics-Informed Learning**: Combine uncertainty with physical constraints
2. **Active Learning**: Use uncertainty to guide data acquisition
3. **Robust Optimization**: Incorporate uncertainty into engineering decisions
4. **Validation Frameworks**: Establish comprehensive testing protocols

This uncertainty quantification framework provides the mathematical foundation and practical tools needed for reliable deployment of deep learning models in electromagnetic field prediction applications where confidence in predictions is crucial for safety and decision-making.